In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cs116-tiep/final_groundtruth.pkl
/kaggle/input/cs116-tiep/01-2025.pkl
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.user_chunk_7.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.user_chunk_1.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_0.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_41.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_42.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_34.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_58.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_49.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purchase_history_daily_chunk_71.parquet
/kaggle/input/cs116-tiep/recommendation dataset/sales_pers.purch

In [2]:
# =============================================================================
# OPTIMIZED NOTEBOOK 1: DATA CLEANING & FEATURE EXTRACTION (FAST VERSION)
# =============================================================================
import polars as pl
import pandas as pd
import numpy as np
import os
import gc
import warnings

warnings.filterwarnings('ignore')

print("=" * 80)
print("🚀 ADVANCED DATA CLEANING - OPTIMIZED FOR SPEED & FEATURES")
print("=" * 80)

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
OLD_DATA_DIR = "/kaggle/input/cs116-tiep/recommendation dataset"
VAL_DATA_PATH = "/kaggle/input/cs116-tiep/01-2025.pkl"
TEST_GT_PATH = "/kaggle/input/cs116-tiep/final_groundtruth.pkl"
OUTPUT_PATH = "/kaggle/working/cleaned_data"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- FILTERING PARAMS ---
MIN_PRICE = 1000              # < 1k VND -> Rác
MAX_PRICE_PERCENTILE = 0.995  # 99.5% -> Loại hàng quá đắt (Outlier)
MIN_QUANTITY = 1
MAX_QUANTITY = 50             # Mua sỉ -> Loại

# Interaction Thresholds
MIN_USER_INTERACTIONS = 5     # User tích cực
MIN_ITEM_INTERACTIONS = 5     # Item phổ biến

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================
def load_parquet_files(file_list):
    if not file_list: return None
    # scan_parquet nhanh hơn read vì nó lazy loading
    return pl.scan_parquet(file_list)

def load_pickle_to_pl(file_path):
    if os.path.exists(file_path):
        print(f"   → Reading pickle: {os.path.basename(file_path)}...")
        try:
            df_pd = pd.read_pickle(file_path)
            # Convert Pandas -> Polars ngay lập tức để giải phóng RAM
            if isinstance(df_pd, pd.DataFrame):
                pl_df = pl.from_pandas(df_pd)
                del df_pd
                gc.collect()
                return pl_df
            return df_pd
        except Exception as e:
            print(f"   ⚠ Error loading pickle: {e}")
    return None

def get_region(province):
    # Mapping chuẩn hóa vùng miền
    province = str(province).lower()
    if any(x in province for x in ['hà nội', 'hanoi', 'hải phòng', 'quảng ninh', 'bắc', 'thái bình', 'nam định', 'hải dương']):
        return 'North'
    elif any(x in province for x in ['hồ chí minh', 'hcm', 'sài gòn', 'bình dương', 'đồng nai', 'vũng tàu', 'long an', 'cần thơ', 'tiền giang']):
        return 'South'
    elif any(x in province for x in ['đà nẵng', 'huế', 'nghệ an', 'thanh hóa', 'quảng', 'bình định', 'khánh hòa']):
        return 'Central'
    else:
        return 'Unknown' # Hoặc map về South nếu muốn fill

# =============================================================================
# 3. LOAD & MERGE RAW DATA
# =============================================================================
print("\n[1/6] LOADING RAW DATA...")

all_files = os.listdir(OLD_DATA_DIR)
item_files = [os.path.join(OLD_DATA_DIR, f) for f in all_files if "item_chunk" in f]
user_files = [os.path.join(OLD_DATA_DIR, f) for f in all_files if "user_chunk" in f]
trans_files = [os.path.join(OLD_DATA_DIR, f) for f in all_files if "purchase_history_daily_chunk" in f]

# Load Trans cũ (Lazy)
lf_trans_old = load_parquet_files(trans_files)
# Load Trans mới (Eager -> convert to Lazy)
df_trans_new_eager = load_pickle_to_pl(VAL_DATA_PATH)

# Chuẩn hóa Schema Trans Cũ
lf_trans_old = lf_trans_old.select([
    pl.col("customer_id").cast(pl.Int32),
    pl.col("item_id").cast(pl.Utf8),
    pl.col("timestamp").cast(pl.Int64),
    pl.col("date_key").cast(pl.Int32),
    pl.col("price").cast(pl.Float64),
    pl.col("quantity").cast(pl.Int32),
])

# Chuẩn hóa Schema Trans Mới & Merge
if df_trans_new_eager is not None:
    # Tạo date_key nếu chưa có
    if "date_key" not in df_trans_new_eager.columns:
         df_trans_new_eager = df_trans_new_eager.with_columns(
            pl.from_epoch("timestamp", time_unit="s").dt.strftime("%Y%m%d").cast(pl.Int32).alias("date_key")
        )
    
    lf_trans_new = df_trans_new_eager.lazy().select([
        pl.col("customer_id").cast(pl.Int32),
        pl.col("item_id").cast(pl.Utf8),
        pl.col("timestamp").cast(pl.Int64),
        pl.col("date_key").cast(pl.Int32),
        pl.col("price").cast(pl.Float64),
        pl.col("quantity").cast(pl.Int32),
    ])
    lf_trans = pl.concat([lf_trans_old, lf_trans_new])
else:
    lf_trans = lf_trans_old

print("   ✓ Transactions schema unified.")

# =============================================================================
# 4. FAST FILTERING (PRICE & NOISE)
# =============================================================================
print("\n[2/6] CALCULATING THRESHOLDS & FILTERING NOISE...")

# 1. Tính Max Price Threshold (Dùng sample 10% để tính cho nhanh thay vì scan all)
# Lưu ý: Fetch 1M dòng đầu để ước lượng phân vị
price_sample = lf_trans.select("price").filter(pl.col("price").is_not_null()).fetch(1000000)
max_price_val = price_sample["price"].quantile(MAX_PRICE_PERCENTILE)

print(f"   -> Max Price Limit (Est.): {max_price_val:,.0f} VND")

# 2. Apply Basic Filter (Lazy)
lf_trans_clean = lf_trans.filter(
    (pl.col("price") >= MIN_PRICE) &
    (pl.col("price") <= max_price_val) &
    (pl.col("quantity") >= MIN_QUANTITY) &
    (pl.col("quantity") <= MAX_QUANTITY)
)

# =============================================================================
# 5. USER & ITEM FEATURE ENGINEERING (FEATURE QUAN TRỌNG)
# =============================================================================
print("\n[3/6] PROCESSING USERS & ITEMS FEATURES...")

# --- A. ITEMS ---
lf_items = load_parquet_files(item_files)
df_items = (
    lf_items
    .filter(pl.col("is_deleted") == False)
    .select([
        pl.col("item_id").cast(pl.Utf8),
        pl.col("price").cast(pl.Float64),
        pl.col("category_l1").fill_null("Unknown"),
        pl.col("category_l2").fill_null("Unknown"),
        pl.col("brand").fill_null("Unknown"),
    ])
    .unique(subset=["item_id"])
    .collect()
)
# Filter Items: Chỉ giữ items có trong transaction sạch (Semi-join logic)
# (Để tiết kiệm, ta sẽ làm bước này sau khi lọc user interactions)

# --- B. USERS (ADD REGION + MEMBERSHIP) ---
lf_users = load_parquet_files(user_files)

# Map Region Function (Optimized with Polars Expressions)
# Ta dùng `map_elements` python sẽ chậm, nên dùng `when-then` nếu có thể, 
# nhưng để đơn giản và đủ nhanh ta dùng map_elements cho cột province duy nhất rồi join lại.

df_users_raw = (
    lf_users
    .filter(pl.col("is_deleted") == False)
    .select([
        pl.col("customer_id").cast(pl.Int32),
        pl.col("province").fill_null("Unknown"),
        pl.col("gender").fill_null("Unknown")
    ])
    .unique(subset=["customer_id"])
    .collect()
)

# 1. Xử lý Region
print("   -> Mapping Regions...")
# Lấy unique provinces để map cho nhanh
df_provinces = df_users_raw.select("province").unique()
df_provinces = df_provinces.with_columns(
    pl.col("province").map_elements(get_region, return_dtype=pl.Utf8).alias("region")
)
# Join region lại bảng user
df_users = df_users_raw.join(df_provinces, on="province", how="left")

# 2. Xử lý Membership Tier (Dựa trên lịch sử mua hàng)
print("   -> Calculating Membership Tiers...")
# Tính tổng chi tiêu mỗi user từ bảng Trans (đã lọc rác cơ bản)
user_stats = (
    lf_trans_clean
    .group_by("customer_id")
    .agg([
        (pl.col("price") * pl.col("quantity")).sum().alias("total_spend"),
        pl.len().alias("transaction_count")
    ])
    .collect()
)

# Phân loại hạng: 
# < 1tr: Standard | 1tr-10tr: Silver | > 10tr: Gold (Ví dụ)
# Hoặc dùng Quantile: Top 20% = Gold
q80 = user_stats["total_spend"].quantile(0.80)
q50 = user_stats["total_spend"].quantile(0.50)

user_stats = user_stats.with_columns(
    pl.when(pl.col("total_spend") >= q80).then(pl.lit("Gold"))
    .when(pl.col("total_spend") >= q50).then(pl.lit("Silver"))
    .otherwise(pl.lit("Standard")).alias("membership_tier")
)

# Join Stats vào User
df_users_final = df_users.join(user_stats, on="customer_id", how="inner")

print(f"   ✓ Users processed: {len(df_users_final):,}")
print(f"   ✓ Membership thresholds: Silver > {q50:,.0f}, Gold > {q80:,.0f}")

# =============================================================================
# 6. INTERACTION FILTERING (THE HEAVY LIFTING)
# =============================================================================
print("\n[4/6] FINAL INTERACTION FILTERING...")

# Chỉ giữ lại Users đạt chuẩn (Interaction >= 5)
valid_users = df_users_final.filter(
    pl.col("transaction_count") >= MIN_USER_INTERACTIONS
)
valid_user_ids = valid_users["customer_id"]

print(f"   -> Retained {len(valid_users):,} users (Min interactions: {MIN_USER_INTERACTIONS})")

# Filter Transactions: Chỉ lấy của Valid Users
lf_trans_user_filtered = lf_trans_clean.filter(
    pl.col("customer_id").is_in(valid_user_ids)
)

# Tính lại Item Counts dựa trên Valid Users
item_counts = (
    lf_trans_user_filtered
    .group_by("item_id")
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") >= MIN_ITEM_INTERACTIONS)
    .collect()
)
valid_item_ids = item_counts["item_id"]

print(f"   -> Retained {len(valid_item_ids):,} items (Min interactions: {MIN_ITEM_INTERACTIONS})")

# Final Filter Transactions
lf_trans_final = lf_trans_user_filtered.filter(
    pl.col("item_id").is_in(valid_item_ids)
)

# Filter Final Items & Users tables to match Transactions
df_items_final = df_items.filter(pl.col("item_id").is_in(valid_item_ids))
df_users_final = valid_users  # Users đã lọc ở trên rồi

# =============================================================================
# 7. SAVE DATA (OPTIMIZED I/O)
# =============================================================================
print("\n[5/6] SAVING DATA...")

# 1. Save Users & Items (Parquet)
df_users_final.write_parquet(os.path.join(OUTPUT_PATH, "users_clean.parquet"))
df_items_final.write_parquet(os.path.join(OUTPUT_PATH, "items_clean.parquet"))

# 2. Save Transactions (PARTITIONED BY MONTH)
# Thay vì lưu từng ngày (rất chậm), ta thêm cột YearMonth và lưu partitioned
# Việc này giúp đọc ghi cực nhanh
print("   -> Saving transactions (Partitioned by Month)...")

# Tính toán final dataframe
df_trans_final = (
    lf_trans_final
    .with_columns(
        (pl.col("date_key") // 100).cast(pl.Int32).alias("month_key") # 20240101 -> 202401
    )
    .collect(streaming=True) # Sử dụng streaming để tránh OOM khi collect lần cuối
)

# Lưu Partitioned (Tự động chia folder theo month_key)
# Kết quả sẽ là: cleaned_data/transactions_partitioned/month_key=202401/part-0.parquet
save_dir = os.path.join(OUTPUT_PATH, "transactions_partitioned")
try:
    df_trans_final.write_parquet(
        save_dir,
        use_pyarrow=True,
        pyarrow_options={"partition_cols": ["month_key"]}
    )
    print("   ✓ Saved partitioned transactions.")
except Exception as e:
    print(f"   ⚠ Partition save failed ({e}). Saving single file...")
    df_trans_final.write_parquet(os.path.join(OUTPUT_PATH, "transactions_clean.parquet"))

# =============================================================================
# 8. SAVE GROUND TRUTH
# =============================================================================
print("\n[6/6] SAVING GROUND TRUTH...")
gt_data = load_pickle_to_pl(TEST_GT_PATH)
if gt_data is not None:
    if isinstance(gt_data, pl.DataFrame):
        gt_data.write_parquet(os.path.join(OUTPUT_PATH, "test_ground_truth.parquet"))
    else:
        pd.to_pickle(gt_data, os.path.join(OUTPUT_PATH, "test_ground_truth.pkl"))

print("\n" + "=" * 80)
print("✅ DONE! DATA IS CLEAN & FEATURE-RICH")
print(f"   - Users: {len(df_users_final):,} (Added: region, membership_tier)")
print(f"   - Items: {len(df_items_final):,}")
print(f"   - Transactions: {len(df_trans_final):,}")
print("=" * 80)

🚀 ADVANCED DATA CLEANING - OPTIMIZED FOR SPEED & FEATURES

[1/6] LOADING RAW DATA...
   → Reading pickle: 01-2025.pkl...
   ✓ Transactions schema unified.

[2/6] CALCULATING THRESHOLDS & FILTERING NOISE...
   -> Max Price Limit (Est.): 975,000 VND

[3/6] PROCESSING USERS & ITEMS FEATURES...
   -> Mapping Regions...
   -> Calculating Membership Tiers...
   ✓ Users processed: 2,563,382
   ✓ Membership thresholds: Silver > 685,800, Gold > 3,582,616

[4/6] FINAL INTERACTION FILTERING...
   -> Retained 1,174,236 users (Min interactions: 5)
   -> Retained 15,972 items (Min interactions: 5)

[5/6] SAVING DATA...
   -> Saving transactions (Partitioned by Month)...
   ✓ Saved partitioned transactions.

[6/6] SAVING GROUND TRUTH...
   → Reading pickle: final_groundtruth.pkl...

✅ DONE! DATA IS CLEAN & FEATURE-RICH
   - Users: 1,174,236 (Added: region, membership_tier)
   - Items: 15,972
   - Transactions: 36,147,990
